# Experiments

In [7]:
# importing libraries
import pandas as pd
from pathlib import Path
import numpy as np

In [2]:
SPLITS = Path("../data/splits")

## Popularity Baseline

In the training set, what are the most rated books? Doesn't matter if they are liked or not, but what are the most rated.

And to check - show this "model" some books. If those books exist in what it "recommends", don't recommend that. And then, check metrics.

In [3]:
train = pd.read_parquet(SPLITS / "train_split.parquet", columns=["user_id", "work_id"])

In [10]:
train.shape

(102089653, 2)

In [4]:
pop = (train.work_id.value_counts().rename("n_interactions").reset_index().rename(columns={"index": "work_id"}))

In [5]:
print(f"{len(pop):,} works ranked")
pop.head(20)

991,014 works ranked


,work_id,n_interactions
0,4640799,311275
1,2792775,286922
2,3212258,231664
3,3275794,209731
4,2402163,193763
5,6231171,193053
6,3046572,185148
7,245494,182440
8,6171458,182049
9,2809203,178722


In [6]:
books = pd.read_parquet("../data/interim/english_works_all.parquet",columns=["work_id", "title", "ratings_total"])

pop.head(20).merge(books, on="work_id", how="left")[["title", "n_interactions", "ratings_total"]]

,title,n_interactions,ratings_total
0,Harry Potter and the Sorcerer's Stone (Harry P...,311275,4970387
1,"The Hunger Games (The Hunger Games, #1)",286922,5064668
2,"Twilight (Twilight, #1)",231664,3991256
3,To Kill a Mockingbird,209731,3399207
4,Harry Potter and the Prisoner of Azkaban (Harr...,193763,2016007
5,Harry Potter and the Chamber of Secrets (Harry...,193053,1950555
6,Harry Potter and the Goblet of Fire (Harry Pot...,185148,1909895
7,The Great Gatsby,182440,2841340
8,"Catching Fire (The Hunger Games, #2)",182049,2013187
9,Harry Potter and the Order of the Phoenix (Har...,178722,1873246


### Note

Yeah, they all look pretty popular.

In [8]:
val = pd.read_parquet(SPLITS / "val_split.parquet",
                      columns=["user_id", "work_id", "rating"])

In [9]:
val.shape

(1805359, 3)

In [11]:
rng = np.random.default_rng(0)
pos = val[val.rating >= 4]
keep = rng.random(len(pos)) < 0.5

fold_in = pd.concat([val[val.rating < 4], pos[keep]])
target = pos[~keep]

seen = fold_in.groupby("user_id").work_id.apply(set)
truth = target.groupby("user_id").work_id.apply(set)

print(f"{len(truth):,} users, median {truth.apply(len).median():.0f} targets")

10,000 users, median 33 targets


In [12]:
K = 10
ranking = pop.work_id.to_numpy()
head = ranking[:K + seen.apply(len).max()]

recs = {u: [b for b in head if b not in seen.get(u, set())][:K]
        for u in truth.index}

In [13]:
def evaluate(recs, truth, k):
    disc = 1 / np.log2(np.arange(2, k + 2))
    rec, ndcg = [], []
    for u, items in recs.items():
        t = truth[u]
        hits = np.array([b in t for b in items[:k]])
        n = min(len(t), k)
        rec.append(hits.sum() / n)
        ndcg.append((hits * disc).sum() / disc[:n].sum())
    return np.mean(rec), np.mean(ndcg)

r, n = evaluate(recs, truth, K)
print(f"recall@{K} {r:.4f}   ndcg@{K} {n:.4f}")

recall@10 0.1977   ndcg@10 0.2214


In [14]:
flat = [b for v in recs.values() for b in v]
print(f"{len(set(flat)):,} distinct books served to {len(recs):,} users")

38 distinct books served to 10,000 users


In [15]:
books.shape

(1002607, 3)

In [21]:
ids = ['16957329', '18967965', '577668', '4208329', '17316785', '25475795', '16943257', '28096028', '25784612', '25487896', '49322086', '23645510', '26021888', '42470883', '13483419', '42916768', '18217235', '18265117', '47207025', '26917226', '25374301', '17438159', '19140994', '25698990', '21557504', '45585402', '42241284', '47202366', '23684134', '3473940', '23912367', '39834830', '21982945', '26655068', '15970676', '21989535', '25490321', '21553745', '48485', '25315302', '11247213', '20734075', '4493959', '39901452', '17847847', '52113307', '2884461', '2018017', '51817045', '22592363', '48548608', '25441722', '86125', '15223263', '42032954', '13009809', '26637637', '45380778', '20654276', '52297984', '40429780', '43193195', '25747377', '49616101', '25432110', '44235417', '43072768', '675039', '18537958', '45125374', '54810218', '50687269', '3147419', '1420167', '44075664', '18231136', '25875621', '44611111', '45432672', '41787820', '16943099', '674566', '26414536', '44689845', '53090016', '6540461', '16474254', '16944948', '4486024', '55254931', '45661159', '14359203', '25656431', '46090993', '18614511', '42744594', '24510920', '17436791', '21765052', '14753465', '49471487', '1216981', '25603448', '3678073', '18714376', '14459349', '14652246', '46726741', '41379824', '55484227', '3136978', '42230353', '54951725', '49565186', '52968527', '46745595', '41833786', '42574399', '24681', '21810158', '44522735', '21946896', '40765364', '41673340', '26054376', '52064078', '40156821', '25733400', '45497310', '42397963', '18104534', '43971101', '21942113', '21509810', '40713246', '752000', '43640002', '26592979', '12782462', '23588933', '182802', '23634692', '24510877', '56476978', '25001710', '25331100', '25687785', '52923852', '52780866', '26019752', '43239836', '51302095', '24978298', '54280550', '45750944', '26571655', '16083149', '17997465', '41695108', '474994', '47251804', '794328', '44729542', '47821964', '21982778', '14698297', '25633272', '41764534', '46725598', '27213845', '44258164', '17264381', '39574', '21494046', '2819058', '15078981', '16165375', '18718456', '6582125', '56812253', '52112295', '1433577', '55918640', '462034', '18447261', '4509184', '24022649', '1819314', '25491154', '48146748', '43033288', '47300899', '15679058', '17114320', '28161920', '23956664', '20268991', '24948181', '26679990', '10318601', '17048672', '51166184', '42902513', '1600810', '4671735', '45335099', '13367959', '13856944', '2960849', '52836672', '21938169', '16971244', '16957670', '20075082', '50298142', '49569346', '50783680', '41262786', '21517184', '901852', '18451904', '15291086', '41818624', '75142', '23903667', '40081395', '13398549', '19089827', '45138191', '25327098', '47464413', '53200576', '45317843', '25387390', '48273176', '16968927', '314286', '21519374', '48015437', '23584221', '14734520', '49650481', '41675232', '43003782', '56331224', '45062201', '42279833', '49036034', '1697235', '19104142', '24661958', '17355362', '46125300', '42216233', '19427632', '18350884', '46141064', '23577210', '49260166', '48072135', '13678080', '42908668', '19010087', '25094135', '48579', '25520368', '24256585', '27477554', '39904276', '17589097', '44045722', '2712336', '50417122', '83137', '36557769', '45990659', '17197563', '15492048', '6139534', '16688330', '44318468', '28954670', '16069312', '18148657', '24942832', '18638515', '46010281', '39900436', '27669927', '40148806', '45957559', '41523914', '44157376', '14682378', '41735805', '9601971', '2628329', '14698686', '1773894', '19107389', '55432897', '47788438', '12807320', '48435762', '42788042', '43056000', '23746462', '54030819', '954483', '27310101', '40816106', '44223824', '24887332', '26488025', '2129532', '42262430', '21133806', '39851998', '22515182', '26866584', '582111', '50480359', '48334566', '41169318', '44473266', '57654951', '37358828', '54984930', '57330295', '3164874', '766752', '51098466', '48044362', '15277681', '24083891', '26232955', '14866656', '15117088', '4193855', '21489166', '45308366', '10924840', '4723869', '1281211', '1833255', '51153814', '45553948', '21995762', '47556144', '13339463', '261596', '52515679', '25959792', '15899123', '19036254', '6774213', '40807866', '21927269', '46011443', '12919241', '21773956', '17040781', '53785735', '25745521', '25485946', '21822450', '48294953', '18325291', '478661', '27690399', '56198849', '7049641', '51892302', '44586535', '42425516', '21466227', '18006220', '1987292', '44860921', '23894131', '18241917', '18350706', '48692790', '42018385', '24929452', '123696', '40690062', '49493057', '1054248', '18570946', '51975581', '6602797', '9225090', '11889537', '40233911', '52702197', '55677833', '18216417', '26172041', '55411381', '21598130', '6149006', '24164880', '53502668', '19180305', '48888797', '27108492', '50285673', '40319123', '53069856', '21370281', '42707664', '42086650', '6146429', '46018432', '15295887', '539957', '25778977', '45729660', '40927545', '25489630', '25465275', '25994868', '54925933', '51421508', '13395719', '56844379', '46099072', '1228778', '26874643', '19176606', '25003948', '1294140', '45686977', '21937515', '14552313', '2037794', '56634096', '54337669', '25388707', '42866891', '23765885', '48039288', '24381783', '3141', '47445251', '21705137', '54671157', '25039729', '19137693', '19086846', '3262727', '20196528', '15378846', '18098253', '24484481', '942833', '37935825', '44512657', '17097804', '2059062', '44093652', '24994620', '19207456', '55146006', '7224496', '2465459', '17992883', '41108310', '18616502', '21978893', '21767531', '244177', '56530474', '48882251', '19107054', '39861283', '25215974', '49029938', '25272014', '19209802', '17952198', '10263164', '19107124', '32198', '2606878', '46115819', '50609201', '26636421', '18107990', '26708491', '26568451', '16679693', '47436425', '7334229', '19011019', '42136572', '44752991', '49285784', '21892248']

hits = books[books.work_id.astype(str).isin(ids)]
hits[["work_id", "title"]]

,work_id,title
962,6582125,Smile
2497,3262727,Fragile Things: Short Fictions and Wonders
3384,17992883,"Just One Year (Just One Day, #2)"
3505,42397963,"Lumberjanes, Vol. 1: Beware the Kitten Holy"
5105,24948181,"Beautiful Beginning (Beautiful Bastard, #3.5)"
...,...,...
694304,47202366,Kitchenelves Revolution
699454,41379824,A Lovely Machine
709163,25489630,A Kick at the Pantry Door
709850,43003782,Sunrises And Other Stories
